<a href="https://colab.research.google.com/github/swapna0727-source/AI/blob/main/Cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("patient_readmission.csv")
X = df[["age", "length_of_stay_days", "num_prior_visits", "num_medications", "has_chronic_condition"]]
y = df["readmitted_30d"]

pipe = Pipeline(steps = [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

for rs in [0, 1, 42, 7, 99]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=rs, stratify=y)
    pipe.fit(Xtr, ytr)
    print("random_state:", rs, "-> test accuracy:", round(pipe.score(Xte, yte), 3))

random_state: 0 -> test accuracy: 0.7
random_state: 1 -> test accuracy: 0.7
random_state: 42 -> test accuracy: 0.8
random_state: 7 -> test accuracy: 0.7
random_state: 99 -> test accuracy: 0.7


In [ ]:
len(yte)

10

In [ ]:
df = pd.read_csv("patient_readmission.csv")
print(df.head())
print(df.shape)
print(df["readmitted_30d"].value_counts())

   patient_id  age  length_of_stay_days  num_prior_visits  num_medications  \
0        7001   79                   14                 4                8   
1        7002   46                    3                 6                9   
2        7003   79                    5                 1                2   
3        7004   27                   10                 3                8   
4        7005   42                   10                 0                9   

   has_chronic_condition  discharge_note_risk_flag  readmitted_30d  
0                      1                         1               1  
1                      0                         0               0  
2                      1                         0               0  
3                      0                         1               1  
4                      1                         1               1  
(40, 8)
readmitted_30d
1    21
0    19
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

X = df[["age", "length_of_stay_days", "num_prior_visits", "num_medications", "has_chronic_condition"]]
y = df["readmitted_30d"]

pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy")
print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std deviation:", scores.std())

Fold scores: [0.5   0.75  0.875 0.625 0.875]
Mean accuracy: 0.725
Std deviation: 0.1457737973711325


In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("Plain KFold - test class balance per fold:")
for i, (train_idx, test_idx) in enumerate(kf.split(X, y)):
    print(f"  fold {i}:", y.iloc[test_idx].value_counts().to_dict())

print("StratifiedKFold - test class balance per fold:")
for i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"  fold {i}:", y.iloc[test_idx].value_counts().to_dict())

Plain KFold - test class balance per fold:
  fold 0: {1: 6, 0: 2}
  fold 1: {1: 4, 0: 4}
  fold 2: {0: 5, 1: 3}
  fold 3: {0: 5, 1: 3}
  fold 4: {1: 5, 0: 3}
StratifiedKFold - test class balance per fold:
  fold 0: {1: 5, 0: 3}
  fold 1: {0: 4, 1: 4}
  fold 2: {0: 4, 1: 4}
  fold 3: {0: 4, 1: 4}
  fold 4: {0: 4, 1: 4}


In [ ]:
for c in [0.01, 0.1, 1, 10, 100]:
    manual_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(C=c, max_iter=1000))])
    manual_scores = cross_val_score(manual_pipe, X, y, cv=skf, scoring="accuracy")
    print(f"C={c}: mean CV accuracy = {manual_scores.mean():.3f}")

C=0.01: mean CV accuracy = 0.650
C=0.1: mean CV accuracy = 0.700
C=1: mean CV accuracy = 0.725
C=10: mean CV accuracy = 0.675
C=100: mean CV accuracy = 0.725


In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
params = {
    "n_estimators":[5,10,15,20,25],
    "max_depth" :[ 2,3,4,5],
    "max_features": [2,3,4]
}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
model = RandomForestClassifier( random_state=42)

In [ ]:
model2 = LogisticRegression()

In [ ]:
model3 = DecisionTreeClassifier(random_state=42)

In [ ]:
grid_model = GridSearchCV( estimator = model,
                          param_grid = params,
                           cv = 5 ,
                           scoring = "accuracy",
                           verbose = 1,
                           n_jobs = -1
                           )

In [ ]:
grid_model.fit(X_train, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5], 'max_features': [2, 3, 4],
                         'n_estimators': [5, 10, 15, 20, 25]},
             scoring='accuracy', verbose=1)

In [ ]:
grid_model.best_estimator_

RandomForestClassifier(max_depth=4, max_features=4, n_estimators=50,
                       random_state=42)

In [ ]:
grid_model.best_score_

np.float64(0.7857142857142857)

In [ ]:
y_pred = grid_model.predict(X_test)

In [ ]:
y_pred

array([1, 0, 0, 0, 1, 0, 1, 0])

In [ ]:
grid_model.score(X_test, y_test)


0.625

In [ ]:
df.columns

Index(['patient_id', 'age', 'length_of_stay_days', 'num_prior_visits',
       'num_medications', 'has_chronic_condition', 'discharge_note_risk_flag',
       'readmitted_30d'],
      dtype='object')

In [ ]:
X_leak = df[["age", "length_of_stay_days", "num_prior_visits",
             "num_medications", "has_chronic_condition", "discharge_note_risk_flag"]]
X_clean = df[["age", "length_of_stay_days", "num_prior_visits",
              "num_medications", "has_chronic_condition"]]
y = df["readmitted_30d"]

scores_leak = cross_val_score(pipe, X_leak, y, cv=skf, scoring="accuracy")
scores_clean = cross_val_score(pipe, X_clean, y, cv=skf, scoring="accuracy")

print("WITH leakage feature    - fold scores:", scores_leak.round(3), "mean:", round(scores_leak.mean(), 3))
print("WITHOUT leakage feature - fold scores:", scores_clean.round(3), "mean:", round(scores_clean.mean(), 3))

WITH leakage feature    - fold scores: [1.    1.    1.    0.875 1.   ] mean: 0.975
WITHOUT leakage feature - fold scores: [0.5   0.75  0.875 0.625 0.875] mean: 0.725


In [ ]:
from sklearn.model_selection import train_test_split

Xtr_leak, Xte_leak, ytr, yte = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
pipe.fit(Xtr_leak, ytr)
print("Single split WITH leak - test accuracy:", pipe.score(Xte_leak, yte))

Xtr_clean, Xte_clean, ytr2, yte2 = train_test_split(X_clean, y, test_size=0.25, random_state=42, stratify=y)
pipe.fit(Xtr_clean, ytr2)
print("Single split WITHOUT leak - test accuracy:", pipe.score(Xte_clean, yte2))

Single split WITH leak - test accuracy: 1.0
Single split WITHOUT leak - test accuracy: 0.8
